# Phase 2 qwen — 실험 결과 종합 분석

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore')

# ── 한국어 폰트 설정 ──
nanum = [f.fname for f in fm.fontManager.ttflist if 'NanumGothic' in f.name]
noto  = [f.fname for f in fm.fontManager.ttflist if 'NotoSansCJK' in f.name or 'Noto Sans CJK' in f.name]

if nanum:
    plt.rcParams['font.family'] = 'NanumGothic'
elif noto:
    plt.rcParams['font.family'] = 'Noto Sans CJK KR'
else:
    print('⚠️ 한국어 폰트를 찾을 수 없습니다. 기본 폰트를 사용합니다.')

plt.rcParams['axes.unicode_minus'] = False

# ── 경로 설정 ──
NOTEBOOK_DIR  = os.path.dirname(os.path.abspath('__file__'))
PROJECT_ROOT  = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..', '..'))
RESULTS_ROOT  = os.path.join(PROJECT_ROOT, 'results', 'phase2_qwen')

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'RESULTS_ROOT : {RESULTS_ROOT}')

# ── 스타일 설정 ──
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# ── 상수 ──
DATASETS = ['humaneval_fin']
METHODS  = ['single','repair','code_then_plan','code_then_plan_repair','rpe_policy','rpe_policy_v2']
METHOD_LABELS = {
    'single'   : 'Single Shot',
    'repair'   : 'Repair Loop',
    'planner_coder' : 'Planner-Coder',
    'code_then_plan' : 'Code-Then-Plan',
    'code_then_replan' : 'Code-Then-Replan',
    'code_then_plan_repair' : 'Code-Then-Plan+Repair',
    'rpe_policy' : 'Policy Loop',
    'rpe_policy_v2': 'Policy Loop v2'
}

# ── 색상 팔레트 ──
METHOD_COLORS = {
    'single'        : '#4C72B0',
    'repair'       : '#55A868',
    'planner_coder'        : '#C44E52',
    'code_then_plan' : '#8172B2',
    'code_then_replan' : '#9EDDFF',
    'code_then_plan_repair' : '#DD8452',
    'rpe_policy' : '#FFA500',
    'rpe_policy_v2': '#FF6347'
}

PROJECT_ROOT : /home/dibaeck/workspace/project_IR_focus_sLM_orchestration
RESULTS_ROOT : /home/dibaeck/workspace/project_IR_focus_sLM_orchestration/results/phase2_qwen


---
## 1. 데이터 로드

In [2]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def load_jsonl(path):
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

# ── 결과 로드 ──
summaries          = {}   # (dataset, method) -> summary dict
analyses           = {}   # (dataset, method) -> analysis dict
step_logs_all      = {}   # (dataset, method) -> list of step records
traj_logs_all      = {}   # (dataset, method) -> list of trajectory records
failure_examples_all = {} # (dataset, method) -> dict of failure examples

found = []
missing = []

for dataset in DATASETS:
    for method in METHODS:
        base = os.path.join(RESULTS_ROOT, dataset, method)
        summary_path = os.path.join(base, 'summary.json')
        if os.path.exists(summary_path):
            key = (dataset, method)
            summaries[key]     = load_json(summary_path)
            # analyses[key]      = load_json(os.path.join(base, 'analysis.json'))
            step_logs_all[key] = load_jsonl(os.path.join(base, 'step_logs.jsonl'))
            traj_logs_all[key] = load_jsonl(os.path.join(base, 'trajectory_logs.jsonl'))
            
            fe_path = os.path.join(base, 'failure_examples.json')
            if os.path.exists(fe_path):
                failure_examples_all[key] = load_json(fe_path)
            
            found.append(f'{dataset}/{method}')
        else:
            missing.append(f'{dataset}/{method}')

print(f'✅ 로드 성공 ({len(found)}개):', ', '.join(found))
if missing:
    print(f'⚠️  결과 없음 ({len(missing)}개):', ', '.join(missing))

✅ 로드 성공 (6개): humaneval_fin/single, humaneval_fin/repair, humaneval_fin/code_then_plan, humaneval_fin/code_then_plan_repair, humaneval_fin/rpe_policy, humaneval_fin/rpe_policy_v2


---
## 2-1. 핵심 지표 비교 (pass@1, exec_success_rate, conditional_pass)

In [3]:
# ── summary DataFrame 구성 ──
rows = []

for (dataset, method), s in summaries.items():
    if s is None:
        continue

    max_calls = s.get("max_calls", 10)
    success_key = s.get("success_metric_name", f"success@{max_calls}")

    row = {
        "dataset": dataset,
        "method": method,
        "method_label": METHOD_LABELS.get(method, method),

        "total": s.get("total_problems", s.get("total", 0)),

        # 기존 num_pass/pass@1 대신 success 계열 사용
        "num_success": s.get("num_success", s.get("success", s.get("num_pass", s.get("passed", 0)))),

        "success_metric_name": success_key,
        "success@k": s.get(
            "success_at_k",
            s.get(success_key, s.get("pass_at_1", s.get("pass@1", 0.0)))
        ),

        "execution_success_rate": s.get("execution_success_rate", 0.0),

        # 기존 conditional_pass와 새 conditional_success 둘 다 대응
        "conditional_success": s.get(
            "conditional_success",
            s.get("conditional_pass", 0.0)
        ),

        "avg_tokens": s.get("avg_tokens", 0.0),
        "avg_latency": s.get("avg_latency", 0.0),
        "avg_calls": s.get("avg_calls", 0.0),
    }

    rows.append(row)

df_summary = pd.DataFrame(rows)
df_summary = df_summary.set_index(["dataset", "method"])

# 방법 순서 고정
method_order = [
    m for m in METHODS
    if m in df_summary.index.get_level_values("method").unique()
]

print("\n=== Summary DataFrame ===")
display(
    df_summary[
        [
            "total",
            "num_success",
            "success_metric_name",
            "success@k",
            "execution_success_rate",
            "conditional_success",
            "avg_tokens",
            "avg_latency",
            "avg_calls",
        ]
    ].round(4)
)


=== Summary DataFrame ===


total  num_success success_metric_name  \
dataset       method                                                          
humaneval_fin single                   164          114           success@1   
              repair                   164          140          success@10   
              code_then_plan           164          150          success@10   
              code_then_plan_repair    164          151          success@10   
              rpe_policy               164          154          success@10   
              rpe_policy_v2            164          151          success@10   

                                     success@k  execution_success_rate  \
dataset       method                                                     
humaneval_fin single                    0.6951                  0.8476   
              repair                    0.8537                  0.8902   
              code_then_plan            0.9146                  0.9939   
              code_then_plan_repair     0.9207                  0.9573   
              rpe_policy                0.9390                  0.9817   
              rpe_policy_v2             0.9207                  0.9939   

                                     conditional_success  avg_tokens  \
dataset       method                                                   
humaneval_fin single                              0.8201    310.6768   
              repair                              0.9589   1841.0671   
              code_then_plan                      0.9202    735.6524   
              code_then_plan_repair               0.9618    908.7988   
              rpe_policy                          0.9565      0.0000   
              rpe_policy_v2                       0.9264      0.0000   

                                     avg_latency  avg_calls  
dataset       method                                         
humaneval_fin single                      2.4059     1.0000  
              repair                      8.0875     2.7073  
              code_then_plan              3.5982     2.2439  
              code_then_plan_repair       4.2419     2.5061  
              rpe_policy                  0.0000     0.0000  
              rpe_policy_v2               0.0000     0.0000

## 2-2. 토큰 비교

In [4]:
# ── method x stage별 call-level token stats ──
stage_token_rows = []

for dataset in DATASETS:
    for method in METHODS:
        key = (dataset, method)
        steps = pd.DataFrame(step_logs_all.get(key, []))

        if steps.empty or 'stage' not in steps.columns:
            continue

        for stage, g in steps.groupby('stage'):
            stage_token_rows.append({
                'dataset': dataset,
                'method': method,
                'method_label': METHOD_LABELS.get(method, method),
                'stage': stage,
                'num_calls': len(g),

                'input_min': g['input_tokens'].min(),
                'input_avg': g['input_tokens'].mean(),
                'input_max': g['input_tokens'].max(),

                'output_min': g['output_tokens'].min(),
                'output_avg': g['output_tokens'].mean(),
                'output_max': g['output_tokens'].max(),

                'total_min': g['total_tokens'].min(),
                'total_avg': g['total_tokens'].mean(),
                'total_max': g['total_tokens'].max(),
            })

df_stage_token_stats = pd.DataFrame(stage_token_rows)
df_stage_token_stats = df_stage_token_stats.set_index(['dataset', 'method', 'stage'])

display(df_stage_token_stats[
    [
        'num_calls',
        'input_min', 'input_avg', 'input_max',
        'output_min', 'output_avg', 'output_max',
        'total_min', 'total_avg', 'total_max',
    ]
].round(2))

num_calls  input_min  \
dataset       method                stage                             
humaneval_fin single                generate         164         38   
              repair                generate         164         38   
                                    repair           280        233   
              code_then_plan        generate         164         38   
                                    plan             102        116   
                                    plan_code        102        183   
              code_then_plan_repair generate         164         38   
                                    plan              96        116   
                                    plan_code         96        183   
                                    repair            55        267   
              rpe_policy            generate         164         38   
                                    plan              68        116   
                                    plan_code         68        183   
                                    repair            70        257   
              rpe_policy_v2         generate         164         38   
                                    plan              48        116   
                                    plan_code         81        210   
                                    repair            24        384   
                                    replan            33        384   

                                               input_avg  input_max  \
dataset       method                stage                             
humaneval_fin single                generate      134.09        391   
              repair                generate      134.09        391   
                                    repair        653.86       1305   
              code_then_plan        generate      134.09        391   
                                    plan          225.54        468   
                                    plan_code     325.50        661   
              code_then_plan_repair generate      134.09        391   
                                    plan          225.04        468   
                                    plan_code     319.10        608   
                                    repair        449.24        941   
              rpe_policy            generate      134.09        391   
                                    plan          363.97       1030   
                                    plan_code     335.18        614   
                                    repair        609.56       1143   
              rpe_policy_v2         generate      134.09        391   
                                    plan          215.85        468   
                                    plan_code     322.78        571   
                                    repair        633.29        925   
                                    replan        569.67       1260   

                                               output_min  output_avg  \
dataset       method                stage                               
humaneval_fin single                generate           14      176.59   
              repair                generate           14      166.55   
                                    repair              4      248.38   
              code_then_plan        generate           14      160.88   
                                    plan               17       66.95   
                                    plan_code          18       90.55   
              code_then_plan_repair generate           14      166.53   
                                    plan               22       61.04   
                                    plan_code          18       88.24   
                                    repair              3      153.89   
              rpe_policy            generate           14      168.84   
                                    plan               17       68.54   
                                    plan_

In [5]:
display(
    df_stage_token_stats.loc[('humaneval_fin', 'rpe_policy')]
    .round(2)
)

# generate   : 초기 single-shot 비용
# plan       : planner prompt 비용
# plan_code  : plan-conditioned coding 비용
# replan     : failure-aware planning 비용
# replan_code: revised-plan coding 비용
# repair     : feedback-based repair 비용

,method_label,num_calls,input_min,input_avg,input_max,output_min,output_avg,output_max,total_min,total_avg,total_max
stage,,,,,,,,,,,
generate,Policy Loop,164,38,134.09,391,14,168.84,512,63,302.93,716
plan,Policy Loop,68,116,363.97,1030,17,68.54,179,150,432.51,1123
plan_code,Policy Loop,68,183,335.18,614,21,96.38,352,257,431.56,950
repair,Policy Loop,70,257,609.56,1143,3,250.89,512,266,860.44,1617


In [6]:
def build_step_success_curve(step_logs_all):
    rows = []

    for (dataset, method), logs in step_logs_all.items():
        df = pd.DataFrame(logs)
        if df.empty:
            continue

        g = df.groupby("step_id")["test_pass"].mean().reset_index()

        for _, r in g.iterrows():
            rows.append({
                "dataset": dataset,
                "method": method,
                "method_label": METHOD_LABELS.get(method, method),
                "step_id": r["step_id"],
                "success_rate": r["test_pass"]
            })

    return pd.DataFrame(rows)

In [7]:
## (1) Step-wise success curve (필수)
# df_step_curve = build_step_success_curve(step_logs_all)

# plt.figure()
# for method in method_order:
#     sub = df_step_curve[df_step_curve["method"] == method]
#     if sub.empty:
#         continue
#     plt.plot(sub["step_id"], sub["success_rate"], label=METHOD_LABELS.get(method, method))

# plt.xlabel("Step")
# plt.ylabel("Success Rate")
# plt.legend()
# plt.title("Step-wise Success Rate")
# plt.show()

In [8]:
## (2) First success step 얼마나 빨리 푸는지
def compute_first_success(step_logs_all):
    rows = []

    for (dataset, method), logs in step_logs_all.items():
        df = pd.DataFrame(logs)
        if df.empty:
            continue

        success_df = df[df["test_pass"] == True]

        first = success_df.groupby("trajectory_id")["step_id"].min()

        rows.append({
            "dataset": dataset,
            "method": method,
            "method_label": METHOD_LABELS.get(method, method),
            "avg_first_success_step": first.mean()
        })

    return pd.DataFrame(rows)

df_first = compute_first_success(step_logs_all)
display(df_first)

,dataset,method,method_label,avg_first_success_step
0,humaneval_fin,single,Single Shot,0.000000
1,humaneval_fin,repair,Repair Loop,0.457143
2,humaneval_fin,code_then_plan,Code-Then-Plan,44.226667
3,humaneval_fin,code_then_plan_repair,Code-Then-Plan+Repair,60.490066
4,humaneval_fin,rpe_policy,Policy Loop,0.876623
5,humaneval_fin,rpe_policy_v2,Policy Loop v2,0.675497


In [9]:
## (3) Policy action distribution
def build_policy_action_stats(step_logs_all):
    rows = []

    for (dataset, method), logs in step_logs_all.items():
        df = pd.DataFrame(logs)
        if df.empty or "policy_action" not in df.columns:
            continue

        dist = df["policy_action"].value_counts(normalize=True)

        for action, ratio in dist.items():
            rows.append({
                "dataset": dataset,
                "method": method,
                "method_label": METHOD_LABELS.get(method, method),
                "action": action,
                "ratio": ratio
            })

    return pd.DataFrame(rows)

df_policy = build_policy_action_stats(step_logs_all)
display(df_policy)

,dataset,method,method_label,action,ratio
0,humaneval_fin,code_then_plan,Code-Then-Plan,generate,0.722826
1,humaneval_fin,code_then_plan,Code-Then-Plan,plan,0.277174
2,humaneval_fin,code_then_plan_repair,Code-Then-Plan+Repair,generate,0.632603
3,humaneval_fin,code_then_plan_repair,Code-Then-Plan+Repair,plan,0.233577
4,humaneval_fin,code_then_plan_repair,Code-Then-Plan+Repair,repair,0.133820
5,humaneval_fin,rpe_policy,Policy Loop,generate,0.443243
6,humaneval_fin,rpe_policy,Policy Loop,plan,0.367568
7,humaneval_fin,rpe_policy,Policy Loop,repair,0.189189
8,humaneval_fin,rpe_policy_v2,Policy Loop v2,generate,0.468571
9,humaneval_fin,rpe_policy_v2,Policy Loop v2,plan,0.274286


---

In [15]:
def build_transition_counts_from_steps(step_logs_all):
    rows = []

    for (dataset, method), logs in step_logs_all.items():
        df = pd.DataFrame(logs)
        if df.empty:
            continue

        df = df.sort_values(["trajectory_id", "step_id"])

        for traj_id, g in df.groupby("trajectory_id"):
            g = g.sort_values("step_id")

            # 상태 정의 (더 informative)
            states = [
                f"{r['stage']}:{r['error_type']}"
                for _, r in g.iterrows()
            ]

            for i in range(len(states) - 1):
                trans = f"{states[i]}->{states[i+1]}"
                rows.append({
                    "dataset": dataset,
                    "method": method,
                    "transition": trans,
                })

    df = pd.DataFrame(rows)

    if df.empty:
        return df

    return df.value_counts(["dataset", "method", "transition"]).reset_index(name="count")

df_trans = build_transition_counts_from_steps(step_logs_all)

In [16]:
iterative_methods = ['repair', 'code_then_plan', 'code_then_plan_repair', 'rpe_policy', 'rpe_policy_v2']

for dataset in DATASETS:
    sub = df_trans[
        (df_trans['dataset'] == dataset) &
        (df_trans['method'].isin(iterative_methods))
    ]

    if sub.empty:
        continue

    pivot = sub.groupby(['method', 'transition'])['count'].sum().unstack(fill_value=0)

    pivot = pivot.reindex([m for m in iterative_methods if m in pivot.index])
    pivot.index = [METHOD_LABELS.get(m, m) for m in pivot.index]

    # 🔥 중요: 비율로 변환
    pivot_ratio = pivot.div(pivot.sum(axis=1), axis=0)

    print(f'\n[{dataset}] Transition Ratio:')
    display(pivot_ratio.style.background_gradient(cmap='Blues', axis=None))


[humaneval_fin] Transition Ratio:


transition,generate:AssertionError->plan:None,generate:AssertionError->repair:AssertionError,generate:AssertionError->repair:NameError,generate:AssertionError->repair:None,generate:AssertionError->repair:SyntaxError,generate:AttributeError->repair:NameError,generate:IndexError->plan:None,generate:IndexError->repair:IndexError,generate:IndexError->repair:None,generate:IndexError->repair:SyntaxError,generate:NameError->plan:None,generate:NameError->repair:AssertionError,generate:NameError->repair:NameError,generate:NameError->repair:None,generate:NameError->repair:SyntaxError,generate:NameError->repair:TypeError,generate:SyntaxError->plan:None,generate:SyntaxError->repair:AssertionError,generate:SyntaxError->repair:NameError,generate:SyntaxError->repair:None,generate:SyntaxError->repair:SyntaxError,generate:TypeError->plan:None,generate:TypeError->repair:None,generate:ValueError->plan:None,generate:ValueError->repair:NameError,generate:ValueError->repair:TypeError,plan:None->plan_code:AssertionError,plan:None->plan_code:AttributeError,plan:None->plan_code:IndentationError,plan:None->plan_code:NameError,plan:None->plan_code:None,plan:None->plan_code:SyntaxError,plan:None->plan_code:TypeError,plan:None->plan_code:ValueError,plan_code:AssertionError->plan:None,plan_code:AssertionError->repair:AssertionError,plan_code:AssertionError->repair:NameError,plan_code:AssertionError->repair:None,plan_code:AssertionError->repair:SyntaxError,plan_code:AssertionError->replan:None,plan_code:AttributeError->repair:AttributeError,plan_code:IndentationError->repair:SyntaxError,plan_code:NameError->plan:None,plan_code:NameError->repair:AssertionError,plan_code:NameError->repair:NameError,plan_code:NameError->repair:SyntaxError,plan_code:NameError->replan:None,plan_code:SyntaxError->plan:None,plan_code:SyntaxError->repair:AssertionError,plan_code:SyntaxError->repair:NameError,plan_code:SyntaxError->repair:SyntaxError,plan_code:SyntaxError->replan:None,plan_code:TypeError->repair:TypeError,plan_code:ValueError->repair:None,repair:AssertionError->plan:None,repair:AssertionError->repair:AssertionError,repair:AssertionError->repair:NameError,repair:AssertionError->repair:None,repair:AssertionError->repair:SyntaxError,repair:AssertionError->replan:None,repair:AttributeError->plan:None,repair:IndentationError->repair:AssertionError,repair:IndentationError->repair:IndentationError,repair:IndentationError->repair:NameError,repair:IndexError->repair:IndexError,repair:IndexError->repair:SyntaxError,repair:KeyError->repair:SyntaxError,repair:NameError->plan:None,repair:NameError->repair:AssertionError,repair:NameError->repair:IndexError,repair:NameError->repair:NameError,repair:NameError->repair:None,repair:NameError->repair:SyntaxError,repair:NameError->replan:None,repair:SyntaxError->plan:None,repair:SyntaxError->repair:AssertionError,repair:SyntaxError->repair:IndentationError,repair:SyntaxError->repair:KeyError,repair:SyntaxError->repair:NameError,repair:SyntaxError->repair:None,repair:SyntaxError->repair:SyntaxError,repair:TypeError->plan:None,repair:TypeError->repair:TypeError,replan:None->plan_code:AssertionError,replan:None->plan_code:NameError,replan:None->plan_code:None,replan:None->plan_code:SyntaxError
Repair Loop,0.000000,0.032143,0.014286,0.028571,0.021429,0.007143,0.000000,0.003571,0.000000,0.000000,0.000000,0.003571,0.003571,0.000000,0.003571,0.000000,0.000000,0.003571,0.000000,0.014286,0.035714,0.000000,0.003571,0.000000,0.003571,0.003571,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.096429,0.028571,0.000000,0.032143,0.000000,0.000000,0.003571,0.003571,0.003571,0.003571,0.007143,0.000000,0.000000,0.007143,0.003571,0.392857,0.014286,0.014286,0.000000,0.000000,0.028571,0.007143,0.000000,0.032143,0.035714,0.075000,0.000000,0.028571,0

In [18]:
success_cols = [c for c in pivot_ratio.columns if 'None' in c or 'PASS' in c]
display(pivot_ratio[success_cols])

# PASS 전이만 보기

transition,generate:AssertionError->plan:None,generate:AssertionError->repair:None,generate:IndexError->plan:None,generate:IndexError->repair:None,generate:NameError->plan:None,generate:NameError->repair:None,generate:SyntaxError->plan:None,generate:SyntaxError->repair:None,generate:TypeError->plan:None,generate:TypeError->repair:None,...,repair:NameError->plan:None,repair:NameError->repair:None,repair:NameError->replan:None,repair:SyntaxError->plan:None,repair:SyntaxError->repair:None,repair:TypeError->plan:None,replan:None->plan_code:AssertionError,replan:None->plan_code:NameError,replan:None->plan_code:None,replan:None->plan_code:SyntaxError
Repair Loop,0.000000,0.028571,0.000000,0.000000,0.000000,0.000000,0.000000,0.014286,0.000000,0.003571,...,0.000000,0.014286,0.000000,0.000000,0.035714,0.000000,0.000000,0.000000,0.000000,0.000000
Code-Then-Plan,0.132353,0.000000,0.009804,0.000000,0.034314,0.000000,0.073529,0.000000,0.004902,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Code-Then-Plan+Repair,0.133603,0.000000,0.000000,0.000000,0.016194,0.000000,0.072874,0.000000,0.000000,0.000000,...,0.036437,0.000000,0.000000,0.036437,0.000000,0.004049,0.000000,0.000000,0.000000,0.000000
Policy Loop,0.111650,0.000000,0.000000,0.004854,0.000000,0.004854,0.000000,0.000000,0.000000,0.004854,...,0.048544,0.014563,0.000000,0.000000,0.000000,0.004854,0.000000,0.000000,0.000000,0.000000
Policy Loop v2,0.139785,0.000000,0.010753,0.000000,0.010753,0.000000,0.000000,0.000000,0.005376,0.005376,...,0.026882,0.000000,0.021505,0.005376,0.000000,0.000000,0.107527,0.021505,0.021505,0.026882


In [20]:
self_loops = [
    c for c in pivot_ratio.columns
    if c.split('->')[0] == c.split('->')[1]
]
display(pivot_ratio[self_loops])

# self-loop (local trap)

transition,repair:AssertionError->repair:AssertionError,repair:IndentationError->repair:IndentationError,repair:IndexError->repair:IndexError,repair:NameError->repair:NameError,repair:SyntaxError->repair:SyntaxError,repair:TypeError->repair:TypeError
Repair Loop,0.096429,0.003571,0.003571,0.392857,0.075000,0.028571
Code-Then-Plan,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Code-Then-Plan+Repair,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Policy Loop,0.029126,0.000000,0.000000,0.043689,0.004854,0.004854
Policy Loop v2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [21]:
def simplify_error(e):
    if e is None:
        return "NONE"
    if "Syntax" in e:
        return "Syntax"
    if "Type" in e:
        return "Type"
    return "Other"